In [ ]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://fantasy.premierleague.com/statistics")
driver.set_window_size(1920, 3000)
driver.execute_script("document.body.style.zoom='25%'")

from selenium.webdriver.common.by import By
driver.find_element(By.XPATH, "//button[text()='Accept All Cookies']").click()
rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
data = []
page = 1
while True:
    print(f"📄 กำลังเก็บข้อมูลหน้า {page} ...")

    rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")

    for row in rows:
        cols = row.find_elements(By.TAG_NAME, "td")
        if len(cols) >= 5:
            player = cols[0].text
            cost = cols[1].text
            selected = cols[2].text
            form = cols[3].text
            points = cols[4].text
            data.append([player, cost, selected, form, points])

    # ลองคลิกปุ่ม Next
    try:
        next_btn = driver.find_element(By.XPATH, "//button[@aria-label='Next']")
        # ถ้าปุ่ม disable จะข้าม
        if not next_btn.is_enabled():
            print("✅ ถึงหน้าสุดท้ายแล้ว")
            break
        next_btn.click()
        time.sleep(0.5)
        page += 1
    except NoSuchElementException:
        print("✅ ไม่พบปุ่ม Next แล้ว — จบการเก็บข้อมูล")
        break

# แปลงเป็น DataFrame
df = pd.DataFrame(data, columns=["Player", "Cost", "Sel%", "Form", "Pts"])
df.to_excel(r"C:\Users\User\Downloads\Raw Data FPL\FPL.xlsx")

In [ ]:
fpl = pd.read_excel(r"C:\Users\User\Downloads\Raw Data FPL\FPL.xlsx")
fpl.head(5)
fpl[['Name Player', 'Club', 'Position']] = fpl['Player'].str.split('\n', expand=True)
fpl = fpl.drop(columns=['Player'])
fpl = fpl.drop(columns=['Unnamed: 0'])
new_order = ['Name Player', 'Club', 'Position'] + [col for col in fpl.columns if col not in ['Name Player', 'Club', 'Position']]
fpl = fpl[new_order]
fpl.to_excel(r"C:\Users\User\Downloads\Raw Data FPL\FPL2.xlsx")